# Prepare data for GEO submission 

In [1]:
options(warn=-1)

In [2]:
library_load <- suppressMessages(
    
    suppressWarnings(
        
        list(

            # Data 
            library(R.utils), 
            library(tidyverse), 
            library(data.table)

        )
    )
)

In [3]:
random_seed <- 42
set.seed(random_seed)

In [4]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20220623HSCT")

# Transfer data 

In [5]:
meta <- read.csv("meta.csv") %>% 
    # Re-set patient id
    dplyr::left_join(., data.frame(patient_name=c(763, 764, 766, 768, 783, 784, 785, 4, 7, 8), patient_id_new=paste0("Patient_", 1:10)), by=join_by(patient_name)) %>% 
    dplyr::mutate(patient_id=patient_id_new) %>% 
    dplyr::mutate(patient_id=ifelse(is.na(patient_id), "Patient_11", patient_id)) %>% 
    # Re-set sample_name
    dplyr::mutate(sample_name=paste0(patient_id, "_", time_point)) %>%
    # Set GEO path 
    dplyr::mutate(geo_path=paste0("/nobackup/peer/fdeckert/FD20220623HSCT/data/GEO/scrnaseq/", tolower(sample_name)))

In [ ]:
overwrite <- TRUE
skip <- FALSE

In [ ]:
for (i in 1:nrow(meta)){
    
    x <- meta[i, , drop=FALSE]
    
    R.utils::copyFile(srcPathname=paste0(x$sample_path, "/raw_feature_bc_matrix/barcodes.tsv.gz"), destPathname=paste0(x$geo_path, "_barcodes_gex.tsv.gz"), overwrite=overwrite, skip=skip)
    R.utils::copyFile(srcPathname=paste0(x$sample_path, "/raw_feature_bc_matrix/features.tsv.gz"), destPathname=paste0(x$geo_path, "_features_gex.tsv.gz"), overwrite=overwrite, skip=skip)
    R.utils::copyFile(srcPathname=paste0(x$sample_path, "/raw_feature_bc_matrix/matrix.mtx.gz"), destPathname=paste0(x$geo_path, "_matrix_gex.mtx.gz"), overwrite=overwrite, skip=skip)
    R.utils::copyFile(srcPathname=paste0(x$sample_path, "/feature_reference.csv"), destPathname=paste0(x$geo_path, "_feature_reference_gex.csv"), overwrite=overwrite, skip=skip)
    R.utils::copyFile(srcPathname=paste0(x$sample_path, "/HTO_demux.csv"), destPathname=paste0(x$geo_path, "_hto_demux.csv"), overwrite=overwrite, skip=skip)
    
    if(grepl("CoR_", x$sample_path)) {x$tcr_path <- gsub("_transcriptome", "_abTCR_VDJ", x$sample_path)}
    if(grepl("GvD_", x$sample_path)) {x$tcr_path <- gsub("_transcriptome", "_abTCR_VDJ", x$sample_path)}
    if(grepl("S_", x$sample_path)) {x$tcr_path <- gsub("_transcriptome", "_VDJ", x$sample_path)}
    if(grepl("HSCT_", x$sample_path)) {x$tcr_path <- gsub("_transcriptome", "_abTCR_VDJ", x$sample_path)}
    
    R.utils::copyFile(srcPathname=paste0(x$tcr_path, "/filtered_contig_annotations.csv"), destPathname=paste0(x$geo_path, "_filtered_contig_annotations_tcr.csv"), overwrite=overwrite, skip=skip)
    
}

# Write meta data

## Samples

In [ ]:
scrnaseq_sample <- data.frame(
    
        sample_name=tolower(rep(meta$sample_name, 2)), 
        patient_id=tolower(rep(meta$patient_id, 2)) %>% factor(., levels=paste0("patient_", 1:11)), 
        time_point=tolower(rep(meta$time_point, 2)) %>% factor(., levels=c("baseline", "tx", "d14", "d100", "gvhd")), 
        library_name=tolower(c(paste0(meta$sample_name, "_gex"), paste0(meta$sample_name, "_tcr"))), 
        title=rep(c("GEX", "TCR"), each=nrow(meta)) %>% factor(., levels=c("GEX", "TCR")), 
        organism="homo sapiens", 
        tissue="blood, skin", 
        cell_type="blood CD45+, skin CD45+/CD45-", 
        library_type=rep(c("mRNA+HTO", "TCR"), each=nrow(meta))
          ) %>% 

    dplyr::mutate(
    
        processed_data_file_1=ifelse(title=="GEX", paste0(sample_name, "_barcodes_gex.tsv.gz"), NA), 
        processed_data_file_2=ifelse(title=="GEX", paste0(sample_name, "_features_gex.tsv.gz"), NA), 
        processed_data_file_3=ifelse(title=="GEX", paste0(sample_name, "_matrix_gex.mtx.gz"), NA), 
        processed_data_file_4=ifelse(title=="GEX", paste0(sample_name, "_hto_demux.csv"), NA), 
        processed_data_file_5=ifelse(title=="TCR", paste0(sample_name, "_filtered_contig_annotations_tcr.csv"), NA), 
        raw_file=paste0(library_name, "_mock.fastq")

    
    ) %>% 

    dplyr::arrange(patient_id, time_point)

In [ ]:
write.csv(scrnaseq_sample, "/nobackup/peer/fdeckert/FD20220623HSCT/data/GEO/scrnaseq_sample.csv")

In [ ]:
file.create(paste0("/nobackup/peer/fdeckert/FD20220623HSCT/data/GEO/mock/", scrnaseq_sample$raw_file))

## Checksum 

In [ ]:
scrnaseq_checksum <- data.frame(
    
    file_name=list.files("/nobackup/peer/fdeckert/FD20220623HSCT/data/GEO/scrnaseq/", full.names=FALSE), 
    file_checksum=tools::md5sum(list.files("/nobackup/peer/fdeckert/FD20220623HSCT/data/GEO/scrnaseq/", full.names=TRUE)) %>% as.vector(), 
    
    file_raw_name=list.files("/nobackup/peer/fdeckert/FD20220623HSCT/data/GEO/mock/", full.names=FALSE), 
    file_raw_checksum=tools::md5sum(list.files("/nobackup/peer/fdeckert/FD20220623HSCT/data/GEO/mock/", full.names=TRUE)) %>% as.vector()
          
          ) %>% 

    dplyr::mutate(
    
        patient_id=paste0("Patient_", lapply(strsplit(file_name, "_"), '[[', 2)) %>% factor(., levels=paste0("Patient_", 1:11)), 
        time_point=lapply(strsplit(file_name, "_"), '[[', 3) %>% factor(., levels=c("baseline", "tx", "d14", "d100", "gvhd"))
    
    ) %>% 

    dplyr::arrange(patient_id, time_point)

In [ ]:
write.csv(scrnaseq_checksum, "/nobackup/peer/fdeckert/FD20220623HSCT/data/GEO/scrnaseq_checksum.csv")